# GameTheory-7-ExtensiveForm (Twin C#)

> **Jumeau C# (.NET Interactive)** de [GameTheory-7-ExtensiveForm.ipynb](GameTheory-7-ExtensiveForm.ipynb) — marathon **#4956** (parite .NET <-> Python), axe-2 SOTA **#3801** Prong B.

Le notebook Python modelise les jeux sous forme extensive (arbre de jeu) avec `networkx` (graphe), affiche les arbres avec `matplotlib`, et invoque `OpenSpiel` (`pyspiel`) pour Kuhn Poker. **Ce twin deroule tout a la main** (BCL .NET 9, 0 NuGet) : on construit la structure de donnees de l'arbre de jeu, on modelise les **ensembles d'information** (infosets) et les **noeuds de nature** (chance), et on code la **conversion forme extensive -> forme normale**. La visualisation de l'arbre se fait en **ASCII** (equivalent du plot matplotlib).

> **Distinct du twin GT-9-C#** : GT-9 resout les jeux extensifs par **induction arriere** (un algorithme de resolution). Ce notebook GT-7 est sur la **representation et la modelisation** (comment encoder un arbre de jeu, des infosets, des noeuds de hasard, et convertir vers la forme normale). Les deux sont complementaires : modeliser (GT-7) puis resoudre (GT-9).

## 1. De la forme normale a la forme extensive

La **forme normale** (GT-2) represente un jeu par sa matrice de gains. La **forme extensive** represente le **deroulement temporel** : qui joue quand, quelles actions sont disponibles, et ce que chaque joueur sait a chaque decision (les **ensembles d'information**). C'est la structure naturelle pour les jeux dynamiques (chacun joue a son tour) et les jeux a information incomplete (chance, informations cachees).

Les trois types de noeuds :
- **Noeud de decision** (`player >= 1`) : un joueur choisit une action parmi `actions`.
- **Noeud terminal** (`player == -1`) : la partie finit, `payoffs` donne le gain de chaque joueur.
- **Noeud de nature / chance** (`player == 0`) : la Nature tire une action selon `chanceProbs` (carte, de, etc.).

In [1]:
// Modele de jeu sous forme extensive (BCL .NET, 0 NuGet)
using System.Globalization;

public class GameNode
{
    public string Id { get; set; }
    public int Player { get; set; }                      // -1 = terminal, 0 = nature, >=1 = joueur
    public List<string> Actions { get; set; } = new();
    public Dictionary<string, GameNode> Children { get; set; } = new();
    public double[] Payoffs { get; set; }                // noeud terminal uniquement
    public string Infoset { get; set; }                  // ensemble d'information
    public Dictionary<string, double> ChanceProbs { get; set; }  // noeud de nature
    public bool IsTerminal => Player == -1;
    public bool IsChance => Player == 0;
}

public class ExtensiveFormGame
{
    public string Name { get; set; }
    public int NumPlayers { get; set; }
    public GameNode Root { get; set; }
    public Dictionary<string, GameNode> Nodes { get; set; } = new();
    public Dictionary<string, List<string>> Infosets { get; set; } = new();

    public ExtensiveFormGame(string name, int numPlayers) { Name = name; NumPlayers = numPlayers; }

    public GameNode AddNode(GameNode n) { Nodes[n.Id] = n; if (n.Player >= 1 && n.Infoset == null) n.Infoset = n.Id; if (n.Infoset != null) { if (!Infosets.ContainsKey(n.Infoset)) Infosets[n.Infoset] = new(); Infosets[n.Infoset].Add(n.Id); } return n; }
    public void SetRoot(GameNode r) { Root = r; AddNode(r); }
    public void AddChild(GameNode parent, string action, GameNode child) { if (!parent.Actions.Contains(action)) parent.Actions.Add(action); parent.Children[action] = child; AddNode(child); }
}

// Helpers (la culture FR ne persiste pas entre cellules .NET Interactive ; -0.0 supprime)
static string FI(double x, string fmt = "F2") { double z = Math.Abs(x) < 1e-12 ? 0.0 : x; return string.Format(CultureInfo.InvariantCulture, "{0:" + fmt + "}", z); }
static void Show(object s) { display(s?.ToString() ?? "(null)"); }
display("Modele ExtensiveFormGame pret : GameNode (decision/terminal/chance + infoset) + arbre + infosets index.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Modele ExtensiveFormGame pret : GameNode (decision/terminal/chance + infoset) + arbre + infosets index.

## 2. Exemple : jeu d'entree sur le marche

Le **jeu d'entree** canonique : un entrant (J1) decide d'entrer (`In`) ou de rester dehors (`Out`). S'il entre, l'incumbent (J2) decide de `Fight` (guerre des prix) ou `Accommodate` (coexistence). C'est un jeu **parfait information** (chacun voit tout).

In [2]:
// Jeu d'entree sur le marche (information parfaite)
var entry = new ExtensiveFormGame("Entry Game", 2);
var root = new GameNode { Id = "entrant", Player = 1, Actions = new() { "In", "Out" } };
entry.SetRoot(root);

var incumbent = new GameNode { Id = "incumbent", Player = 2, Actions = new() { "Fight", "Accommodate" } };
entry.AddChild(root, "In", incumbent);
entry.AddChild(root, "Out", new GameNode { Id = "out", Player = -1, Payoffs = new[] { 0.0, 2.0 } });

entry.AddChild(incumbent, "Fight", new GameNode { Id = "fight", Player = -1, Payoffs = new[] { -1.0, -1.0 } });
entry.AddChild(incumbent, "Accommodate", new GameNode { Id = "accommodate", Player = -1, Payoffs = new[] { 1.0, 1.0 } });

Show(entry.Name + " : " + entry.Nodes.Count + " noeuds, information parfaite (1 infoset par decision).");

Entry Game : 5 noeuds, information parfaite (1 infoset par decision).

### Rendu ASCII de l'arbre

Le Python utilise `matplotlib` + `networkx` pour dessiner l'arbre. Nous le rendons en **ASCII** : un parcours recursif avec indentation par profondeur, qui marque les noeuds de nature (N), les infosets (~), et les payoffs terminaux.

In [3]:
// Rendu ASCII de l'arbre de jeu (equivalent du plot matplotlib)
static string RenderTree(ExtensiveFormGame g)
{
    var sb = new System.Text.StringBuilder();
    void Rec(GameNode n, string lastAction, int depth)
    {
        string ind = new string(' ', depth * 2);
        string branch = depth == 0 ? "" : (lastAction.Length > 0 ? "[" + lastAction + "] " : "");
        if (n.IsTerminal)
            sb.AppendLine(ind + branch + "* (J1=" + FI(n.Payoffs[0]) + ", J2=" + FI(n.Payoffs[1]) + ")");
        else if (n.IsChance)
            sb.AppendLine(ind + branch + "NATURE" + (n.Infoset != null ? " ~" + n.Infoset : ""));
        else
            sb.AppendLine(ind + branch + "J" + n.Player + (n.Infoset != null ? " ~" + n.Infoset : ""));
        foreach (var a in n.Actions)
            Rec(n.Children[a], a, depth + 1);
    }
    sb.AppendLine(g.Name);
    sb.AppendLine(new string('-', g.Name.Length));
    Rec(g.Root, "", 0);
    return sb.ToString();
}
Show(RenderTree(entry));

Entry Game
----------
J1 ~entrant
  [In] J2 ~incumbent
    [Fight] * (J1=-1.00, J2=-1.00)
    [Accommodate] * (J1=1.00, J2=1.00)
  [Out] * (J1=0.00, J2=2.00)


### Interpretation : structure du jeu d'entree

L'arbre montre la chronologie : J1 (entrant) choisit `In`/`Out` ; s'il entre, J2 (incumbent) choisit `Fight`/`Accommodate`. Les payoffs terminaux revelent la structure : `Fight` (-1,-1) est un <b>equilibre non-credible</b> (la menace de guerre n'est pas credible une fois l'entree effective), `Accommodate` (1,1) est l'issue rationnelle. C'est precisement ce que l'<b>induction arriere</b> (GT-9) formalise.

## 3. Strategies dans les jeux extensifs

Une **strategie pure** en forme extensive est un **plan d'action complet** : pour chaque ensemble d'information du joueur, quelle action jouer. Ce n'est pas "une action", c'est un plan contingents (une action PAR infoset). Le nombre de strategies pures d'un joueur = produit du nombre d'actions sur chacun de ses infosets.

In [4]:
// Enumeration des strategies pures d'un joueur = plans contingents (produit cartesien sur ses infosets)
static List<Dictionary<string, string>> PureStrategies(ExtensiveFormGame g, int player)
{
    // Infosets de ce joueur, conservant l'ordre d'action de chaque noeud
    var playerInfosets = g.Infosets
        .Where(kv => g.Nodes[kv.Value[0]].Player == player)
        .Select(kv => (id: kv.Key, actions: g.Nodes[kv.Value[0]].Actions.ToList()))
        .ToList();
    var plans = new List<Dictionary<string, string>>();
    void Rec(int idx, Dictionary<string, string> cur)
    {
        if (idx == playerInfosets.Count) { plans.Add(new Dictionary<string, string>(cur)); return; }
        var (iid, acts) = playerInfosets[idx];
        foreach (var a in acts) { cur[iid] = a; Rec(idx + 1, cur); }
    }
    Rec(0, new Dictionary<string, string>());
    return plans;
}

// Entry game : J1 a 1 infoset (entrant: In/Out = 2 strategies) ; J2 a 1 infoset (incumbent: Fight/Accommodate = 2 strategies)
var s1 = PureStrategies(entry, 1);
var s2 = PureStrategies(entry, 2);
Show("J1 strategies (" + s1.Count + ") : " + string.Join("  ", s1.Select(p => "{" + string.Join(",", p.Values) + "}")));
Show("J2 strategies (" + s2.Count + ") : " + string.Join("  ", s2.Select(p => "{" + string.Join(",", p.Values) + "}")));

J1 strategies (2) : {In}  {Out}

J2 strategies (2) : {Fight}  {Accommodate}

## 4. Information parfaite vs imparfaite

La difference cruciale : en **information parfaite**, chaque infoset est un singleton (le joueur sait exactement ou il est). En **information imparfaite**, un infoset regroupe plusieurs noeuds : le joueur sait qu'il est a l'un d'entre eux mais ne sait pas lequel.

Le jeu **mouvement simultane** (Matching Pennies en forme extensive) : J1 joue Pile/Face, puis J2 joue Pile/Face **sans voir** le coup de J1. Les deux noeuds de decision de J2 sont dans le **meme infoset** `j2_choice`.

In [5]:
// Mouvement simultane = Matching Pennies en forme extensive (information imparfaite)
var sim = new ExtensiveFormGame("Simultaneous Move (Matching Pennies)", 2);
var r = new GameNode { Id = "j1", Player = 1, Actions = new() { "Pile", "Face" } };
sim.SetRoot(r);
// J2 ne voit pas le coup de J1 -> les deux noeuds J2 partagent l'infoset "j2_choice"
var j2pile = new GameNode { Id = "j2_after_pile", Player = 2, Actions = new() { "Pile", "Face" }, Infoset = "j2_choice" };
var j2face = new GameNode { Id = "j2_after_face", Player = 2, Actions = new() { "Pile", "Face" }, Infoset = "j2_choice" };
sim.AddChild(r, "Pile", j2pile);
sim.AddChild(r, "Face", j2face);
// Payoffs (Matching Pennies : J1 gagne si meme face, J2 sinon)
sim.AddChild(j2pile, "Pile", new GameNode { Id = "pp", Player = -1, Payoffs = new[] { 1.0, -1.0 } });
sim.AddChild(j2pile, "Face", new GameNode { Id = "pf", Player = -1, Payoffs = new[] { -1.0, 1.0 } });
sim.AddChild(j2face, "Pile", new GameNode { Id = "fp", Player = -1, Payoffs = new[] { -1.0, 1.0 } });
sim.AddChild(j2face, "Face", new GameNode { Id = "ff", Player = -1, Payoffs = new[] { 1.0, -1.0 } });

Show(RenderTree(sim));
Show("Infosets : " + sim.Infosets.Count + " (j1 singleton, j2_choice regroupe 2 noeuds = J2 ignore le coup de J1).");
Show("=> Le marqueur ~j2_choice sur 2 noeuds = c'est l'ENCODAGE de l'information imparfaite.");

Simultaneous Move (Matching Pennies)
------------------------------------
J1 ~j1
  [Pile] J2 ~j2_choice
    [Pile] * (J1=1.00, J2=-1.00)
    [Face] * (J1=-1.00, J2=1.00)
  [Face] J2 ~j2_choice
    [Pile] * (J1=-1.00, J2=1.00)
    [Face] * (J1=1.00, J2=-1.00)


Infosets : 2 (j1 singleton, j2_choice regroupe 2 noeuds = J2 ignore le coup de J1).

=> Le marqueur ~j2_choice sur 2 noeuds = c'est l'ENCODAGE de l'information imparfaite.

## 5. Jeux avec hasard : noeuds de nature

Quand le hasard est implique (tirage de carte, de), on ajoute des **noeuds de nature** (`player == 0`) avec des probabilites sur chaque branche. Le jeu de carte simple : la Nature tire High/Low (p=0.5), J1 voit la carte et Bet/Check, puis si Bet J2 Call/Fold **sans voir la carte** (infoset partage).

In [6]:
// Jeu de carte simple avec noeud de nature + infoset (information incomplete)
var card = new ExtensiveFormGame("Simple Card Game", 2);
var nature = new GameNode { Id = "nature", Player = 0, Actions = new() { "High", "Low" }, ChanceProbs = new() { { "High", 0.5 }, { "Low", 0.5 } } };
card.SetRoot(nature);

// J1 voit la carte (infosets distincts High/Low car J1 connait sa carte)
var j1h = new GameNode { Id = "j1_high", Player = 1, Actions = new() { "Bet", "Check" }, Infoset = "j1_H" };
var j1l = new GameNode { Id = "j1_low", Player = 1, Actions = new() { "Bet", "Check" }, Infoset = "j1_L" };
card.AddChild(nature, "High", j1h);
card.AddChild(nature, "Low", j1l);

// J2 apres Bet ne sait pas High/Low -> MEME infoset "j2_bet"
var j2hb = new GameNode { Id = "j2_h_bet", Player = 2, Actions = new() { "Call", "Fold" }, Infoset = "j2_bet" };
var j2lb = new GameNode { Id = "j2_l_bet", Player = 2, Actions = new() { "Call", "Fold" }, Infoset = "j2_bet" };
card.AddChild(j1h, "Bet", j2hb); card.AddChild(j1h, "Check", new GameNode { Id = "h_check", Player = -1, Payoffs = new[] { 1.0, 1.0 } });
card.AddChild(j1l, "Bet", j2lb); card.AddChild(j1l, "Check", new GameNode { Id = "l_check", Player = -1, Payoffs = new[] { -1.0, -1.0 } });
// Payoffs Bet+Fold=(1,-1) ; Bet+Call depend High(2,-2)/Low(-2,2)
card.AddChild(j2hb, "Call", new GameNode { Id = "h_call", Player = -1, Payoffs = new[] { 2.0, -2.0 } });
card.AddChild(j2hb, "Fold", new GameNode { Id = "h_fold", Player = -1, Payoffs = new[] { 1.0, -1.0 } });
card.AddChild(j2lb, "Call", new GameNode { Id = "l_call", Player = -1, Payoffs = new[] { -2.0, 2.0 } });
card.AddChild(j2lb, "Fold", new GameNode { Id = "l_fold", Player = -1, Payoffs = new[] { 1.0, -1.0 } });

Show(RenderTree(card));
Show("Infosets : " + string.Join(", ", card.Infosets.Keys) + " | j2_bet regroupe J2_h_bet + J2_l_bet (J2 ignore la carte).");

Simple Card Game
----------------
NATURE
  [High] J1 ~j1_H
    [Bet] J2 ~j2_bet
      [Call] * (J1=2.00, J2=-2.00)
      [Fold] * (J1=1.00, J2=-1.00)
    [Check] * (J1=1.00, J2=1.00)
  [Low] J1 ~j1_L
    [Bet] J2 ~j2_bet
      [Call] * (J1=-2.00, J2=2.00)
      [Fold] * (J1=1.00, J2=-1.00)
    [Check] * (J1=-1.00, J2=-1.00)


Infosets : j1_H, j1_L, j2_bet | j2_bet regroupe J2_h_bet + J2_l_bet (J2 ignore la carte).

## 6. OpenSpiel : jeux extensifs avances (section conceptuelle)

Le notebook Python invoque **OpenSpiel** (`pyspiel`) pour charger Kuhn Poker, un jeu classique a information incomplete. **OpenSpiel n'a pas d'equivalent .NET** (pas de binding .NET officiel) — c'est un verdict **INTRINSIC** pour cette section specifique (#3801) : on ne peut pas re-invoquer OpenSpiel en C#.

Ce twin documente donc la **structure de Kuhn Poker** (3 cartes, infosets par carte + historique d'encheres) et renvoie au twin Python pour l'execution OpenSpiel. Le coeur du notebook (modelisation + conversion, sections 1-5+7) est **from-scratch SOTA-OK** : ce que networkx/OpenSpiel cachent (l'arbre, les infosets, le hasard), ce twin le rend explicite.

**Kuhn Poker (structure)** : 2 joueurs, 3 cartes (J,Q,K) distribuees par la Nature, chaque joueur voit sa carte (infoset par carte), mises `Pass`/`Bet`. Information incomplete (la carte de l'adversaire est cachee). C'est le banc d'essai canonique du **CFR** (GT-13) et des solveurs de poker modernes.

## 7. Conversion forme extensive -> forme normale

Tout jeu extensif peut etre converti en jeu **normal** (matrice de gains) : on enumere les strategies pures de chaque joueur (plans contingents, section 3), puis pour chaque couple (s1, s2) on calcule le payoff espere en traversant l'arbre (les noeuds de nature etant ponderes par leurs probabilites). La matrice resultat est exactement le jeu normal equivalent.

In [7]:
// Conversion extensive -> normale : matrice de gains sur les strategies pures
static double[] ExpectedPayoff(ExtensiveFormGame g, Dictionary<int, Dictionary<string, string>> stratByPlayer)
{
    // Traversee recursive : a chaque noeud, choisir l'action du joueur (via son infoset) ou ponderer la nature
    double[] Rec(GameNode n)
    {
        if (n.IsTerminal) return n.Payoffs;
        if (n.IsChance)
        {
            double[] acc = new double[g.NumPlayers];
            foreach (var (a, p) in n.ChanceProbs)
            {
                var sub = Rec(n.Children[a]);
                for (int k = 0; k < g.NumPlayers; k++) acc[k] += p * sub[k];
            }
            return acc;
        }
        // Noeud de decision : l'action = strategie du joueur pour l'infoset de ce noeud
        string action = stratByPlayer[n.Player][n.Infoset];
        return Rec(n.Children[action]);
    }
    return Rec(g.Root);
}

static string ConvertToNormal(ExtensiveFormGame g)
{
    var s1 = PureStrategies(g, 1);
    var s2 = PureStrategies(g, 2);
    var sb = new System.Text.StringBuilder();
    sb.AppendLine("Forme normale equivalente : " + g.Name);
    sb.AppendLine(new string('-', 50));
    sb.AppendLine("J1 \\ J2  " + string.Join("    ", s2.Select(p => "{" + string.Join(",", p.Values) + "}")));
    foreach (var a in s1)
    {
        var strat = new Dictionary<int, Dictionary<string, string>> { { 1, a } };
        var cells = new List<string>();
        foreach (var b in s2)
        {
            strat[2] = b;
            var pay = ExpectedPayoff(g, strat);
            cells.Add("(" + FI(pay[0]) + "," + FI(pay[1]) + ")");
        }
        sb.AppendLine("{" + string.Join(",", a.Values) + "}  " + string.Join("  ", cells));
    }
    return sb.ToString();
}

Show("=== Entry Game -> forme normale ===");
Show(ConvertToNormal(entry));
Show("=== Simultaneous Move -> forme normale (= Matching Pennies) ===");
Show(ConvertToNormal(sim));

=== Entry Game -> forme normale ===

Forme normale equivalente : Entry Game
--------------------------------------------------
J1 \ J2  {Fight}    {Accommodate}
{In}  (-1.00,-1.00)  (1.00,1.00)
{Out}  (0.00,2.00)  (0.00,2.00)


=== Simultaneous Move -> forme normale (= Matching Pennies) ===

Forme normale equivalente : Simultaneous Move (Matching Pennies)
--------------------------------------------------
J1 \ J2  {Pile}    {Face}
{Pile}  (1.00,-1.00)  (-1.00,1.00)
{Face}  (-1.00,1.00)  (1.00,-1.00)


### Interpretation : equivalence des formes

La conversion de l'**Entry Game** donne une matrice 2x2 dont l'equilibre est `(In, Accommodate)` — la meme conclusion que l'induction arriere (GT-9), mais obtenue par la theorie normale. La conversion du **Simultaneous Move** redonne exactement la matrice de **Matching Pennies** (pas de point-selle, strategies mixtes uniformes). C'est la preuve que les deux formes sont **equivalentes** en information complete : tout jeu extensif se reduit a une matrice de gains.

## 8. Resume

| Concept | Python (twin) | Twin C# (ici) |
|---------|---------------|----------------|
| Arbre de jeu | `networkx` (graphe) | **structure `GameNode` recursive from-scratch** |
| Visualisation | `matplotlib` | **rendu ASCII** (parcours + indentation) |
| OpenSpiel / Kuhn Poker | `pyspiel` (execution) | **section conceptuelle** (INTRINSIC, pas de binding .NET) |
| Conversion -> normale | `itertools.product` | **enumeration recursive des plans contingents** |

**Ce que la lib cache et que ce twin rend explicite** : la structure recursive du noeud (decision/terminal/chance + infoset), le **codage de l'information imparfaite par infoset partage** (le coeur conceptuel), le **ponderation des noeuds de nature** dans le calcul de payoff espere, et le **produit cartesien des plans contingents** qui definit une strategie pure en forme extensive.

> **Lien avec GT-9** : ce notebook modelise la forme extensive (representation). GT-9-C# la **resout** par induction arriere. Les deux se completent : modeliser puis resoudre.

> **Lien avec la formalisation Lean** : les jeux extensifs et les equilibres (SPE) sont formalises dans `game_theory_lean`. Ce notebook en est le versant **computationnel**.

## 9. Exercices

> **Convention** : stubs a completer (`null` + `// TODO etudiant`), jamais d'erreur volontaire — le notebook s'execute de bout en bout meme non complete.

In [8]:
// Exercice 1 : Construire un arbre de jeu (Chain-Store paradox simplifie)
// Etape 1 : 1 entrant, incumbent Fight/Accommodate. Etape 2 : ajouter un 2e entrant qui observe l'issue du 1er.
// Indice : reutiliser AddChild + infosets. Le 2e entrant a-t-il un infoset distinct apres Fight vs Accommodate ?
var ex1 = (ExtensiveFormGame)null;  // votre Chain-Store a 2 entrants
Show("Exercice 1 a completer : construisez un Chain-Store a 2 entrants, affichez RenderTree.");

Exercice 1 a completer : construisez un Chain-Store a 2 entrants, affichez RenderTree.

In [9]:
// Exercice 2 : Convertir le Card Game (section 5) en forme normale
// Indice : appelez ConvertToNormal(card). Combien de strategies pures pour J1 (infosets j1_H, j1_L) ?
//          Pour J2 (infoset unique j2_bet) ? Verifiez que la matrice reflete l'information incomplete.
var ex2 = (string)null;  // = ConvertToNormal(card)
Show("Exercice 2 a completer : ConvertToNormal(card) -> observer la matrice et le compte de strategies.");

Exercice 2 a completer : ConvertToNormal(card) -> observer la matrice et le compte de strategies.

In [10]:
// Exercice 3 : Ajouter un noeud de nature au jeu d'entree (entrant a un cout aleatoire)
// Indice : au noeud 'entrant', le cout d'entree est High(p=0.3) ou Low(p=0.7), modifiant les payoffs 'In'.
// Question : comment l'incumbent doit-il raisonner sans connaitre le cout (nouvel infoset) ?
var ex3 = (ExtensiveFormGame)null;  // entry game avec cout aleatoire
Show("Exercice 3 a completer : ajoutez un noeud de nature cout, discutez l'infoset de l'incumbent.");

Exercice 3 a completer : ajoutez un noeud de nature cout, discutez l'infoset de l'incumbent.

---

## Conclusion

Ce twin a deroule les **cinq couches** de la forme extensive :
1. **Modele** : `GameNode` (decision/terminal/chance + infoset) et `ExtensiveFormGame`.
2. **Arbre** : construction recursive + **rendu ASCII** (equivalent matplotlib).
3. **Strategies** : enumeration des **plans contingents** (produit cartesien sur les infosets).
4. **Information imparfaite** : encodage par **infoset partage** (Matching Pennies, Card Game).
5. **Conversion -> normale** : traversee recursive avec **ponderation des noeuds de nature**.

La ou `networkx`/`OpenSpiel` fournissent des structures opaques, ce notebook montre **le noeud recursif**, **le codage de l'information imparfaite**, **le hasard equilibre** et **le produit cartesien des plans**. C'est toute la mecanique de modelisation, rendue explicite — l'objectif du marathon #4956.

**Prochaines etapes** : une fois le jeu modelise (ce notebook), on le **resout** par induction arriere ([GameTheory-9-BackwardInduction-Csharp](GameTheory-9-BackwardInduction-Csharp.ipynb)) ou par CFR pour l'information incomplete ([GameTheory-13-ImperfectInfo-CFR](GameTheory-13-ImperfectInfo-CFR.ipynb)).

*See #4956 (marathon parite .NET/Python). Refs #3801 (axe-2 SOTA).*

---

*Genere par myia-po-2023, cycle 57, marathon #4956.*